# DD-PRiSM-plus — Step 1: set up and fetch all data

Run once in a **CPU** session (preprocessing needs no GPU, and GPU quota is
only ~30 h/week). Click **Save Version** at the end or everything here is lost.

**Session options (right-hand `<` panel):**
- **Accelerator → None**
- **Internet → On**  ← nothing downloads without this

In [ ]:
# 0. Confirm the session is set up correctly
import subprocess, os

ok = subprocess.run(['curl','-sI','--max-time','15','https://github.com'],
                    capture_output=True).returncode == 0
print('internet:', 'ON' if ok else 'OFF  <-- enable it in Session options, then rerun')
print('free disk:', subprocess.run(['df','-h','/kaggle/working'],
      capture_output=True, text=True).stdout.splitlines()[-1])

## 1. Get the code

In [ ]:
REPO = '/kaggle/working/ddprism-plus'

if os.path.exists(REPO):
    !cd {REPO} && git pull --quiet
else:
    !git clone --quiet https://github.com/SanaNiroomand/DD-PRiSM-plus.git {REPO}

os.chdir(REPO)
print('working in', os.getcwd())

## 2. Install what Kaggle lacks

`zipfile-deflate64` is **mandatory**: DOSERESP.zip uses Deflate64 and the
standard library cannot decompress it.

In [ ]:
!pip install --quiet zipfile-deflate64 rdkit openpyxl
print('installed')

## 3. Check the model code (23 tests, ~4 s)

Proves the vectorised model still matches the published one. If these fail,
stop — do not train.

In [ ]:
!python -m pytest tests -q

## 4. Download the source data (~1 GB)

Files land **directly in `DATA`** — no `Raw` subfolder.

The two DepMap files come from figshare, which has been returning
**202 Accepted** with an empty body site-wide. If that is still happening the
script gives up on the host quickly and defers both, rather than burning your
session on backoff. Everything else still downloads.

In [ ]:
DATA = '/kaggle/working/data'
!python scripts/get_data.py --dest {DATA} --include-optional --attempts 4

## 5. Retry any stragglers

Only needed if step 4 reported failures. Safe to rerun as often as you like —
files already present are skipped, nothing is re-downloaded.

In [ ]:
!python scripts/get_data.py --dest {DATA} --only depmap_expression depmap_samples --attempts 6

## 6. Verify

Every required row must read `ok`. This check has caught four separate silent
failures — truncated files, empty bodies, throttle responses — so do not skip it.

In [ ]:
!python scripts/get_data.py --dest {DATA} --check
print()
!ls -la {DATA} && du -sh {DATA}

## 7. Save it

**Save Version → Save & Run All (Commit).**

That freezes `/kaggle/working` into a versioned output. The next notebook
attaches it via **Add Input → Your Work → Notebook Output**, where it appears
read-only under `/kaggle/input/…` and does *not* count against the 20 GB quota.

Without this step everything here disappears when the session ends.

---

**Next:** preprocessing. Success is landing on exactly **7,915,900** NCI60
training rows and **1,387,317** combination rows.